In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "STIXGeneral", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 7, "axes.labelsize": 7, "axes.titlesize": 7.5,
    "xtick.labelsize": 6, "ytick.labelsize": 6,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "legend.fontsize": 6,
})

BLUE, ORANGE, GREEN = "#4477AA", "#EE7733", "#009988"

# MVAcaller 输出：barcode 以 EMPTY 开头的行即模拟纯 ambient 液滴（零假设集合）
TSV = "/Users/apple/Downloads/benchmarks/sal/test_111_rep1.barcodes.tsv"
df = pd.read_csv(TSV, sep="\t")
empty = df[df["barcode"].str.startswith("EMPTY")].reset_index(drop=True)
n = len(empty)
P_COLS = ["pvalue_rrnapct", "pvalue_ko", "pvalue_non_rrna"]
LABELS = ["rRNA-fraction test", "KEGG-module test", "non-rRNA composition test"]
p = empty[P_COLS].to_numpy(float)
pc = empty["pvalue_combined"].to_numpy(float)

# 对依赖结构稳健的两种 P 值合并方式
W = -2.0 * np.log(np.clip(p, 1e-300, 1.0))
sw = W.sum(axis=1)
mean_w = 6.0
var_w = sw.var(ddof=1)
c_b, df_b = var_w / (2 * mean_w), 2 * mean_w ** 2 / var_w
p_brown = stats.chi2.sf(sw / c_b, df=df_b)
t = np.tan(np.pi * (0.5 - np.clip(p, 1e-15, 1 - 1e-15)))
p_acat = 0.5 - np.arctan(t.mean(axis=-1)) / np.pi

# QQ 图的理论分位与 95% KS 置信带
q = (np.arange(1, n + 1) - 0.5) / n
qx = -np.log10(q)
D95 = 1.3581 / np.sqrt(n)
band_lo = -np.log10(np.clip(q + D95, 0.0, 1.0))
band_hi = -np.log10(np.clip(q - D95, 1e-12, 1.0))

# 三个检验两两之间的 Spearman 相关
pairs = ((0, 1), (0, 2), (1, 2))
pair_labels = [("rRNA-fraction P", "KEGG-module P"),
               ("rRNA-fraction P", "non-rRNA composition P"),
               ("KEGG-module P", "non-rRNA composition P")]
rhos = {(i, j): stats.spearmanr(p[:, i], p[:, j])[0] for i, j in pairs}

ks_c = stats.kstest(pc, "uniform")
fpr = {name: [(pv < a).mean() for a in (0.1, 0.05, 0.01)]
       for name, pv in (("Fisher", pc), ("Brown", p_brown), ("ACAT", p_acat))}
print(f"null barcodes: {n}")


In [ ]:
fig = plt.figure(figsize=(7.09, 5.6))
gs = fig.add_gridspec(3, 3, wspace=0.38, hspace=0.72)

# 第 0 行：三个检验的 QQ 图（a | b | c）
ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1])
ax_c = fig.add_subplot(gs[0, 2])
# 第 1 行：三个两两 hexbin（d | e | f）
ax_d = fig.add_subplot(gs[1, 0])
ax_e = fig.add_subplot(gs[1, 1])
ax_f = fig.add_subplot(gs[1, 2])
# 第 2 行：Fisher 合并 QQ + FPR 曲线，第 3 列留空（g | h | .）
ax_g = fig.add_subplot(gs[2, 0])
ax_h = fig.add_subplot(gs[2, 1])


def qq(ax, x, title, ylabel=False):
    ax.fill_between(qx, band_lo, band_hi, color="0.88", lw=0)
    ax.plot(qx, qx, "k--", lw=0.7)
    ax.plot(qx, -np.log10(np.sort(x)), ".", ms=2.2, color=BLUE, rasterized=True)
    ksp = stats.kstest(x, "uniform").pvalue
    ax.set_title(f"{title}\nKS $P$ = {ksp:.2f}")
    ax.set_xlabel("Expected $-\\log_{10}P$")
    if ylabel:
        ax.set_ylabel("Observed $-\\log_{10}P$")
    lim = qx.max() * 1.06
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)


qq(ax_a, p[:, 0], LABELS[0], ylabel=True)
qq(ax_b, p[:, 1], LABELS[1], ylabel=True)
qq(ax_c, p[:, 2], LABELS[2], ylabel=True)
qq(ax_g, pc, "Fisher-combined $P$", ylabel=True)

# 三个 hexbin 共用同一颜色范围
hb_handles = []
for ax, ((i, j), (xl, yl)) in zip((ax_d, ax_e, ax_f), zip(pairs, pair_labels)):
    hb = ax.hexbin(p[:, i], p[:, j], gridsize=22, bins="log", cmap="viridis",
                   mincnt=1, linewidths=0)
    hb_handles.append(hb)
    rho = rhos[(i, j)]
    ax.set_title(f"$\\rho$ = {rho:+.2f}", fontweight="bold" if abs(rho) > 0.05 else "normal")
    ax.set_xlabel(xl)
    ax.set_ylabel(yl)

hb_max = max(h.norm.vmax for h in hb_handles
             if h.norm is not None and h.norm.vmax is not None)
for h in hb_handles:
    h.set_clim(vmax=hb_max)
cb = fig.colorbar(hb_handles[1], ax=[ax_d, ax_e, ax_f], shrink=0.80, pad=0.02)
cb.set_label("barcodes", fontsize=6)
cb.ax.tick_params(labelsize=5.5, width=0.6)

# FPR 曲线：Fisher 先画（不透明、zorder 低），Brown 与 ACAT 以 alpha=0.55 覆盖其上
aa = (0.1, 0.05, 0.01)
spec_order = (("Fisher", BLUE, "o", 4.2, 1.0, 3),
              ("Brown", ORANGE, "s", 4.0, 0.55, 4),
              ("ACAT", GREEN, "^", 4.0, 0.55, 5))
for name, col, mk, ms, alpha, zo in spec_order:
    pv = {"Fisher": pc, "Brown": p_brown, "ACAT": p_acat}[name]
    ax_h.plot(aa, [fpr[name][k] for k in range(3)], mk, ms=ms, color=col,
              mfc=col, mec="black", mew=0.5, lw=0, alpha=alpha, label=name, zorder=zo)
ax_h.plot((1e-3, 0.1), (1e-3, 0.1), "k--", lw=0.7, zorder=1)
ax_h.set_xscale("log")
ax_h.set_yscale("log")
ax_h.set_xlim(7e-4, 0.16)
ax_h.set_ylim(3e-3, 0.2)
ax_h.set_xlabel("Nominal $\\alpha$")
ax_h.set_ylabel("Observed false-positive rate")
ax_h.legend(frameon=False, loc="upper left", handletextpad=0.2, borderaxespad=0.3)
ax_h.text(0.97, 0.04, "no false positives\nat $\\alpha \\leq 10^{-3}$",
          transform=ax_h.transAxes, ha="right", va="bottom", fontsize=6)

# 面板字母
for ax, letter in zip((ax_a, ax_b, ax_c, ax_d, ax_e, ax_f, ax_g, ax_h), "abcdefgh"):
    ax.annotate(letter, xy=(-0.14, 1.10), xycoords="axes fraction",
                fontsize=9, fontweight="bold", ha="right", va="bottom")

plt.show()

# caption 数字
print("--- caption numbers ---")
for (i, j), lab in zip(pairs, pair_labels):
    print(f"Spearman {lab[0]} vs {lab[1]}: {rhos[(i, j)]:+.3f}")
print(f"KS combined: stat={ks_c.statistic:.4f}, p={ks_c.pvalue:.3f}")
print(f"Brown scaling: c={c_b:.3f}, df'={df_b:.2f}")
for name in ("Fisher", "Brown", "ACAT"):
    print(f"FPR {name}: " + "  ".join(f"a={a:g}:{fpr[name][k]:.3f}"
                                      for k, a in enumerate((0.1, 0.05, 0.01))))
for j, c in enumerate(P_COLS):
    print(f"KS {c}: p={stats.kstest(p[:, j], 'uniform').pvalue:.3f}")
print(f"n null droplets: {n}")
